In [ ]:
# ILASTIK DEBUG TOOLS
# standalone diagnostic cells — run dependencies + config first, then any check below

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
import h5py
import os
import glob
import json
from bioio import BioImage

# UPDATE THESE
BASE_DIR = r"path/to/base/directory"
PREDICT_DIR = r"path/to/protein/folder"
TRAINING_DIR = os.path.join(BASE_DIR, "vessel_classifier")

PROTEINS = ['CD44', 'SDC1', 'WGA', 'HABP', 'ZO-1', 'eNOS', 'ICAM-1']
MOUSE_TYPES = ['YOUNG', 'TBI', 'OLD', 'AD']
BRAIN_REGIONS = ['PFC', 'HIPP', 'MID']
TARGET_MOUSE_TYPES = ['YOUNG', 'TBI', 'OLD', 'AD']
TARGET_REGIONS = ['PFC', 'HIPP', 'MID']
VESSEL_CHANNEL = 0
MICROGLIA_CHANNEL = 1

def detect_mouse_type(filename):
    fname = filename.upper()
    for mt in MOUSE_TYPES:
        if f'_{mt}_' in fname or fname.startswith(f'{mt}_'):
            return mt
    return 'UNKNOWN'

def detect_region(filename):
    fname = filename.upper()
    for r in BRAIN_REGIONS:
        if r in fname:
            return r
    return 'UNKNOWN'

def is_target(filename):
    mt = detect_mouse_type(filename)
    reg = detect_region(filename)
    return (mt in TARGET_MOUSE_TYPES) and (reg in TARGET_REGIONS)

def norm(img):
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn) if mx > mn else np.zeros_like(img)

def make_green(gray):
    rgb = np.zeros((*gray.shape, 3), dtype=np.uint8)
    rgb[..., 1] = (gray * 255).astype(np.uint8)
    return rgb

def get_channel_config(czi_path):
    img = BioImage(str(czi_path))
    ome = img.ome_metadata
    channels = ome.images[0].pixels.channels
    config = {'protein': 0, 'TL': 0, 'dextran': 0, 'DAPI': 0}
    for i, ch in enumerate(channels):
        ex = float(ch.excitation_wavelength) if ch.excitation_wavelength else 0
        fl = (ch.fluor or '').lower()
        ch_num = i + 1
        if 480 <= ex <= 495 or 'alexa fluor 488' in fl:
            config['TL'] = ch_num
        elif 510 <= ex <= 565 or 'alexa fluor 555' in fl:
            config['protein'] = ch_num
        elif 590 <= ex <= 640 or 'texas red' in fl:
            config['dextran'] = ch_num
        elif 400 <= ex <= 410 or 'dapi' in fl:
            config['DAPI'] = ch_num
    return config

In [ ]:
# CHECK: CHANNEL / LASER ASSIGNMENT + IMAGE METADATA

import xml.etree.ElementTree as ET

def identify_channel_role(excitation, fluor):
    ex = float(excitation) if excitation else 0
    fl = (fluor or '').lower()
    if 480 <= ex <= 495 or '488' in fl or 'alexa fluor 488' in fl:
        return 'TL'
    elif 510 <= ex <= 565 or '555' in fl or '514' in fl:
        return 'protein'
    elif 590 <= ex <= 640 or 'texas red' in fl:
        return 'dextran'
    elif 400 <= ex <= 410 or 'dapi' in fl:
        return 'DAPI'
    return f'?({ex:.0f}nm)'


def get_czi_metadata(fpath):
    img = BioImage(fpath)
    meta = img.metadata

    objective = ''
    tile_scan = False
    acq_date = ''

    # collect track wavelengths and dyes from TrackSetup
    track_wavelengths = []  # ordered list of (track_name, wavelength_nm)
    current_track = None

    # collect attenuation and voltage values in order (not tied to TrackSetup)
    all_attenuation = []
    all_voltage = []

    for elem in meta.iter():
        tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        text = (elem.text or '').strip()
        attribs = elem.attrib

        if tag == 'ObjectiveName' and text:
            objective = text
        elif tag == 'NominalMagnification' and text and not objective:
            objective = text + 'x'
        elif tag == 'LensNA' and text and objective and 'NA' not in objective:
            objective += f' NA={text}'

        if tag == 'TileScanInfo' or (tag == 'AcquisitionMode' and 'tile' in text.lower()):
            tile_scan = True

        if tag in ('CreationDate', 'AcquisitionDateAndTime') and text and not acq_date:
            acq_date = text[:10]

        # track setup: wavelengths
        if tag == 'TrackSetup' and 'Name' in attribs:
            current_track = attribs['Name']
        if current_track and tag == 'Wavelength' and text:
            wl = float(text)
            if wl < 1: wl = wl * 1e9
            track_wavelengths.append((current_track, wl))
            current_track = None  # only take first wavelength per track

        # attenuation and voltage (appear in channel/detector sections, in order)
        if tag == 'Attenuation' and text:
            all_attenuation.append(float(text))
        if tag == 'Voltage' and text:
            all_voltage.append(float(text))

    if not acq_date:
        ome = img.ome_metadata
        if ome.images and ome.images[0].acquisition_date:
            acq_date = str(ome.images[0].acquisition_date)[:10]

    # match protein track (514/561nm) to its index, then pull attenuation/voltage by index
    prot_power, prot_gain = '?', '?'
    for idx, (tname, wl) in enumerate(track_wavelengths):
        if 510 <= wl <= 565:
            if idx < len(all_attenuation):
                prot_power = f'{all_attenuation[idx]*100:.1f}%'
            if idx < len(all_voltage):
                prot_gain = f'{all_voltage[idx]:.0f}'
            break

    return objective or '?', tile_scan, acq_date or '?', prot_power, prot_gain


rows = []
for protein in PROTEINS:
    protein_dir = os.path.join(BASE_DIR, protein)
    if not os.path.exists(protein_dir):
        continue
    for fpath in sorted(glob.glob(os.path.join(protein_dir, '**', '*.czi'), recursive=True)):
        fname = os.path.basename(fpath)
        if not is_target(fname):
            continue
        img = BioImage(fpath)
        dims = img.dims.order
        shape = img.data.shape
        z = shape[dims.index('Z')] if 'Z' in dims else 1
        y = shape[dims.index('Y')] if 'Y' in dims else shape[-2]
        x = shape[dims.index('X')] if 'X' in dims else shape[-1]
        ome = img.ome_metadata
        channels = ome.images[0].pixels.channels

        ch_map = {}
        for i, ch in enumerate(channels):
            role = identify_channel_role(ch.excitation_wavelength, ch.fluor)
            ex = f"{float(ch.excitation_wavelength):.0f}" if ch.excitation_wavelength else '?'
            ch_map[f'C{i+1}'] = f"{role}({ex})"

        objective, tile_scan, acq_date, prot_power, prot_gain = get_czi_metadata(fpath)

        flag = []
        if z > 18: flag.append('Z>18')
        if tile_scan: flag.append('TILE')

        row = {
            'Protein': protein,
            'Filename': fname,
            'X': x,
            'Y': y,
            'Z': z,
            'C1': ch_map.get('C1', '-'),
            'C2': ch_map.get('C2', '-'),
            'C3': ch_map.get('C3', '-'),
            'C4': ch_map.get('C4', '-'),
            'Objective': objective,
            'Tile': 'Yes' if tile_scan else '',
            'Date': acq_date,
            'Laser %': prot_power,
            'PMT V': prot_gain,
            'Flag': ' '.join(flag),
        }
        rows.append(row)
df = pd.DataFrame(rows)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 40)
display(df)

flagged = df[df['Flag'] != '']
if len(flagged):
    print(f"\n{len(flagged)} flagged files:")
    display(flagged[['Protein', 'Filename', 'Flag']])

In [ ]:
# OPTIONAL: export metadata table to CSV

csv_path = os.path.join(BASE_DIR, "image_metadata_report.csv")
df.to_csv(csv_path, index=False)
print(f"Saved to {csv_path}")

In [ ]:
# CHECK: EXPORTED H5 FILES
# run after Step 1 to verify all z-stacks exported correctly

h5_files = sorted(glob.glob(os.path.join(TRAINING_DIR, '*_TL_zstack.h5')))
print(f"{len(h5_files)} h5 files in {TRAINING_DIR}\n")

bad = []
for fpath in h5_files:
    fname = os.path.basename(fpath)
    size_mb = os.path.getsize(fpath) / 1e6
    with h5py.File(fpath, 'r') as f:
        shape = f['data'].shape
    print(f"  {fname:<50} {size_mb:>6.1f} MB  shape={shape}")
    if shape[0] != 18:
        print(f"    [!] Z={shape[0]} != 18")
        bad.append(fname)

if bad:
    print(f"\n{len(bad)} problematic files:")
    for f in bad:
        print(f"  {f}")


In [ ]:
# find target czis for this folder
czi_files = sorted(glob.glob(os.path.join(PREDICT_DIR, '*.czi')))
target_czis = [f for f in czi_files if is_target(os.path.basename(f))]

# CHECK: PROBABILITY FILES
# run after Step 3 to verify ilastik outputs and find missing/corrupted files

prob_dir = os.path.join(PREDICT_DIR, "ilastik_prob")
if not os.path.exists(prob_dir):
    print(f"[!] {prob_dir} doesn't exist. Run Step 3 first.")
else:
    prob_files = sorted(os.listdir(prob_dir))
    print(f"{len(prob_files)} files in {prob_dir}\n")

    good, bad = [], []
    for fname in prob_files:
        fpath = os.path.join(prob_dir, fname)
        size_mb = os.path.getsize(fpath) / 1e6
        with h5py.File(fpath, 'r') as f:
            shape = f['exported_data'].shape
        print(f"  [OK] {fname:<55} {size_mb:>6.1f} MB  shape={shape}")
        good.append(fname)

    # check for missing targets
    target_stems = {os.path.splitext(os.path.basename(f))[0] for f in target_czis}
    predicted_stems = {f.replace('_TL_zstack_prob.h5', '').replace('_prob.h5', '') for f in good}
    missing = target_stems - predicted_stems

    if bad:
        print(f"\n{len(bad)} corrupted (delete and re-run Step 3):")
        for f in bad:
            print(f"  {f}")
    if missing:
        print(f"\n{len(missing)} missing predictions:")
        for m in sorted(missing):
            print(f"  {m}")


In [ ]:
# CHECK: COLORED MASK EXPORT FOR ONE IMAGE
# run after Step 4 to inspect classification quality in detail
# change stem to whichever image you want to examine

stem = "ZO1_AD_PFC_2"  # <-- change this

tl_mip = tifffile.imread(os.path.join(TRAINING_DIR, f"{stem}_TL_MIP.tif")).astype(np.float32)
with h5py.File(os.path.join(PREDICT_DIR, "ilastik_prob", f"{stem}_TL_zstack_prob.h5"), 'r') as f:
    prob = f['exported_data'][:]

vp = prob[..., VESSEL_CHANNEL].astype(np.float32)
mp = prob[..., MICROGLIA_CHANNEL].astype(np.float32)

vessel_seg = np.max(np.argmax(prob, axis=-1) == VESSEL_CHANNEL, axis=0)
microglia_seg = np.max(np.argmax(prob, axis=-1) == MICROGLIA_CHANNEL, axis=0)
vessel_prob_2d = np.max(vp, axis=0)
micro_wins = (mp > vp).astype(np.float32)

# load z-stack for dominant-class
with h5py.File(os.path.join(TRAINING_DIR, f"{stem}_TL_zstack.h5"), 'r') as f:
    tl_zstack = f['data'][:].astype(np.float32)
wta = np.max(tl_zstack * vp * (1.0 - micro_wins), axis=0)

def make_magenta(gray):
    rgb = np.zeros((*gray.shape, 3), dtype=np.uint8)
    rgb[..., 0] = (gray * 255).astype(np.uint8)
    rgb[..., 2] = (gray * 255).astype(np.uint8)
    return rgb

def make_white(gray):
    val = (gray * 255).astype(np.uint8)
    return np.stack([val, val, val], axis=-1)

images = [
    ('Original TL',        make_green(norm(tl_mip))),
    ('Vessel seg (binary)', make_green(vessel_seg.astype(float))),
    ('Microglia seg',      make_magenta(microglia_seg.astype(float))),
    ('Vessel probability', make_white(norm(vessel_prob_2d))),
    ('Dominant-class',   make_green(norm(wta))),
]

fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for ax, (title, rgb) in zip(axes, images):
    ax.imshow(rgb); ax.set_title(title, fontsize=10); ax.axis('off')
plt.suptitle(stem, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Vessel seg: {100*vessel_seg.mean():.1f}%")
print(f"Microglia seg: {100*microglia_seg.mean():.1f}%")
print(f"Overlap: {100*(vessel_seg & microglia_seg).mean():.1f}%")

In [ ]:
# CHECK: THRESHOLD TUNING
# run after Step 3 to compare different vessel probability thresholds
# useful if the weighted images look too dim or too noisy

prob_dir = os.path.join(PREDICT_DIR, "ilastik_prob")
prob_files = sorted(glob.glob(os.path.join(prob_dir, '*_prob.h5')))

if not prob_files:
    print("No probability files. Run Step 3 first.")
else:
    # use first available file
    with h5py.File(prob_files[0], 'r') as f:
        prob = f['exported_data'][:]
    stem = os.path.basename(prob_files[0]).replace('_TL_zstack_prob.h5','').replace('_prob.h5','')

    vp = prob[..., VESSEL_CHANNEL].astype(np.float32)
    mp = prob[..., MICROGLIA_CHANNEL].astype(np.float32)
    if vp.max() > 1: vp = vp / vp.max()
    if mp.max() > 1: mp = mp / mp.max()

    microglia_2d = np.max(mp >= 0.5, axis=0)

    tl_path = os.path.join(TRAINING_DIR, f"{stem}_TL_MIP.tif")
    tl = tifffile.imread(tl_path).astype(np.float32) if os.path.exists(tl_path) else None

    ths = [0.3, 0.4, 0.5, 0.6, 0.7]
    fig, axes = plt.subplots(1, len(ths)+2, figsize=(4*(len(ths)+2), 4))

    if tl is not None:
        p1, p99 = np.percentile(tl, [1, 99.5])
        axes[0].imshow(tl, cmap='gray', vmin=p1, vmax=p99)
    axes[0].set_title('TL MIP'); axes[0].axis('off')

    axes[1].imshow(microglia_2d, cmap='Reds')
    axes[1].set_title(f'Microglia ({100*microglia_2d.mean():.1f}%)'); axes[1].axis('off')

    for j, th in enumerate(ths):
        v_any = np.max(vp >= th, axis=0) if vp.ndim == 3 else (vp >= th)
        m = v_any & ~microglia_2d
        axes[j+2].imshow(m, cmap='gray')
        axes[j+2].set_title(f'T={th} ({100*m.mean():.1f}%)'); axes[j+2].axis('off')

    plt.suptitle(f'Threshold comparison: {stem}', fontsize=10)
    plt.tight_layout(); plt.show()